In [ ]:
import os
import sys
from os.path import join

import pandas as pd
import torch
from Bio import SeqIO
from rdkit import Chem
from rdkit.Chem import AllChem

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

## 1.read

In [ ]:
# queryname = "Arabidopsis_thaliana"
# queryname = "Erigeron_breviscapus"
queryname = "Glycine_max"
# queryname = "Zea_mays"
deletedata = pd.read_pickle(
    join(
        CURRENT_DIR, "..", "data", "our_data/screening2", f"{queryname}_deletedata.pkl"
    )
)
sublist = deletedata["substrate"].unique().tolist()

In [ ]:
len(sublist)

In [ ]:
# ! python distinct.py --input "/home/hanxd/Repositories/ESP/data/our_data/screening2/{queryname}.pep" --output "/home/hanxd/Repositories/ESP/data/our_data/screening2/{queryname}_q.pep"

In [ ]:
enzyme_list = []
sequence_list = []
substrate_list = []

for record in SeqIO.parse(
    f"/home/hanxd/Repositories/ESP/data/our_data/screening2/{queryname}_q.fasta",
    "fasta",
):
    for i in sublist:
        substrate_list.append(i)
        enzyme_list.append(record.id)
        sequence_list.append(str(record.seq))

for record in SeqIO.parse(
    f"/home/hanxd/Repositories/ESP/data/our_data/screening2/{queryname}_q.pep", "fasta"
):
    for i in sublist:
        substrate_list.append(i)
        enzyme_list.append(record.id)
        sequence_list.append(str(record.seq))
df = pd.DataFrame(
    {"enzyme": enzyme_list, "sequence": sequence_list, "substrate": substrate_list}
)

In [ ]:
df

In [ ]:
df = df.drop_duplicates(keep="first")
df = df.drop_duplicates(subset=["enzyme", "substrate"])

In [ ]:
df

In [ ]:
# ! python extract.py esm1b_t33_650M_UR50S {our_data}/screening2/{queryname}_q.pep {our_data}/screening1/esm --repr_layers 33 --include mean

In [ ]:
df["ESM1b"] = ""
df["ECFP"] = ""

rep_dict = join(CURRENT_DIR, "..", "data", "our_data", "screening1/esm/")

for ind in df.index:
    esms = torch.load(rep_dict + str(df["enzyme"][ind]) + ".pt")
    sdf_file_path = our_data + "screening1/SDF_300/" + df["substrate"][ind] + ".sdf"
    mol = Chem.MolFromMolFile(sdf_file_path)
    ecfpso = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024).ToBitString()
    df["ESM1b"][ind] = esms["mean_representations"][33].tolist()
    df["ECFP"][ind] = ecfpso
df["Binding"] = 0

In [ ]:
df.to_pickle(our_data + "screening2/" + f"{queryname}_data.pkl")

In [ ]:
df["substrate"].nunique()

In [ ]:
df["enzyme"].nunique()